# GutBrainIE 2026 Data Exploration

This notebook mirrors the project EDA commands:

```bash
make eda
make eda-all
```

It generates and inspects dataset statistics for the GutBrainIE 2026 course project subtasks:

- **T611 / Task 6.1.1**: Named Entity Recognition
- **T621 / Task 6.2.1**: Mention-Level Relation Extraction

The notebook deliberately reuses the project backend code instead of duplicating data-loading logic.

## 1. Setup

Set the project root, import the reusable data/statistics helpers, and define the dataset/report paths.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from gutbrainie.data.dataset import load_split, build_validation_report
from gutbrainie.data.annotations import deduplicate_entities
from gutbrainie.evaluation.report import generate_data_statistics

DATA_ROOT = PROJECT_ROOT / "data" / "gutbrainie2026"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
DEFAULT_QUALITIES = ("gold", "dev")
ALL_QUALITIES = ("gold", "silver", "silver_2025", "bronze", "dev")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root exists: {DATA_ROOT.exists()} -> {DATA_ROOT}")
print(f"Reports directory: {REPORTS_DIR}")

## 2. Check Available Dataset Files

The local dataset should contain train article files, dev annotations, and test articles. Train annotations are grouped by quality (`gold`, `silver`, `silver_2025`, `bronze`).

In [ ]:
article_files = sorted((DATA_ROOT / "Articles" / "csv_format").glob("*.csv"))
dev_annotation_files = sorted((DATA_ROOT / "Annotations" / "Dev" / "csv_format").glob("*.csv"))
test_files = sorted((DATA_ROOT / "Test_Data").glob("*.csv"))

print("Article CSV files:")
for path in article_files:
    print("-", path.relative_to(PROJECT_ROOT))

print("\nDev annotation CSV files:")
for path in dev_annotation_files:
    print("-", path.relative_to(PROJECT_ROOT))

print("\nTest CSV files:")
for path in test_files:
    print("-", path.relative_to(PROJECT_ROOT))

## 3. Generate EDA Outputs

This cell is the Python equivalent of:

```bash
make eda EDA_QUALITIES="gold dev"
make eda-all
```

Generated files are written under `outputs/reports/`.

In [ ]:
eda_gold_dev = generate_data_statistics(DATA_ROOT, REPORTS_DIR, DEFAULT_QUALITIES)
eda_all = generate_data_statistics(DATA_ROOT, REPORTS_DIR, ALL_QUALITIES)

print("Default EDA qualities:", eda_gold_dev["qualities"])
print("All EDA qualities:", eda_all["qualities"])
print("\nGenerated files:")
for file_path in eda_all["files"]:
    print("-", Path(file_path).relative_to(PROJECT_ROOT))

## 4. Split-Level Statistics

These numbers are useful for the report: document counts, annotation counts, average text lengths, entity/relation density, and class-imbalance indicators.

<!-- notebook-output-images -->

### Saved split-level figures

These static figures are generated under `notebooks/outputs/` and are useful when the notebook is viewed without executing the Plotly cells.

![Annotation volume by split](outputs/01_annotation_volume_by_split.png)

![Average annotations per article](outputs/02_average_annotations_per_article.png)

![Average text length in tokens](outputs/03_average_text_length_tokens.png)

![Class imbalance ratios on log scale](outputs/04_class_imbalance_ratios_log.png)

![Majority label share](outputs/05_majority_label_share.png)


In [ ]:
stats_frames = []
for quality in ALL_QUALITIES:
    path = REPORTS_DIR / f"data_stats_{quality}.csv"
    if path.exists():
        stats_frames.append(pd.read_csv(path))

split_stats = pd.concat(stats_frames, ignore_index=True)
split_stats

In [ ]:
plot_columns = ["documents", "entities", "mention_level_relations", "full_relations"]
long_stats = split_stats.melt(id_vars="split", value_vars=plot_columns, var_name="count_type", value_name="count")
fig = px.bar(
    long_stats,
    x="split",
    y="count",
    color="count_type",
    barmode="group",
    title="GutBrainIE split sizes",
)
fig.show()

In [ ]:
density_columns = ["avg_entities_per_article", "avg_relations_per_article"]
density = split_stats.melt(id_vars="split", value_vars=density_columns, var_name="metric", value_name="value")
px.bar(density, x="split", y="value", color="metric", barmode="group", title="Annotation density per article").show()

## 5. Entity Label Distribution

Entity labels are highly imbalanced. This supports reporting **micro-F1** as the primary score while also inspecting macro-F1/per-label results.

<!-- notebook-output-images -->

### Saved entity-label figures

![Entity label share heatmap](outputs/06_entity_label_share_heatmap.png)

![Gold vs dev entity label shares](outputs/08_gold_vs_dev_entity_label_shares.png)


In [ ]:
entity_dist = pd.read_csv(REPORTS_DIR / "entity_label_distribution.csv")
entity_dist.sort_values(["split", "count"], ascending=[True, False]).head(30)

In [ ]:
entity_totals = entity_dist.groupby("label", as_index=False)["count"].sum().sort_values("count", ascending=False)
px.bar(entity_totals, x="label", y="count", title="Entity label distribution across all inspected splits").show()

In [ ]:
entity_by_split = entity_dist.pivot_table(index="label", columns="split", values="count", fill_value=0, aggfunc="sum")
entity_by_split[sorted(entity_by_split.columns)].sort_values("gold", ascending=False) if "gold" in entity_by_split.columns else entity_by_split

## 6. Relation Predicate Distribution

For T621, predicate imbalance is even more important because relation extraction also depends on correct entity mentions.

<!-- notebook-output-images -->

### Saved relation-label figures

![Relation predicate share heatmap](outputs/07_relation_predicate_share_heatmap.png)

![Gold vs dev relation predicate shares](outputs/09_gold_vs_dev_relation_predicate_shares.png)


In [ ]:
relation_dist = pd.read_csv(REPORTS_DIR / "relation_label_distribution.csv")
relation_dist.sort_values(["split", "count"], ascending=[True, False]).head(40)

In [ ]:
relation_totals = relation_dist.groupby("predicate", as_index=False)["count"].sum().sort_values("count", ascending=False)
px.bar(relation_totals, x="predicate", y="count", title="Relation predicate distribution across all inspected splits").show()

In [ ]:
relation_by_split = relation_dist.pivot_table(index="predicate", columns="split", values="count", fill_value=0, aggfunc="sum")
relation_by_split[sorted(relation_by_split.columns)].sort_values("gold", ascending=False) if "gold" in relation_by_split.columns else relation_by_split

## 7. Relation Triple Distribution

The T621 macro labels can be treated as `(subject_label, predicate, object_label)` triples. These triples are useful for error analysis because the same predicate may behave differently for different entity-type pairs.

<!-- notebook-output-images -->

### Saved relation-triple figure

![Top relation triples in gold](outputs/10_top_relation_triples_gold.png)


In [ ]:
triple_dist = pd.read_csv(REPORTS_DIR / "relation_triple_distribution.csv")
triple_dist.sort_values(["split", "count"], ascending=[True, False]).head(50)

In [ ]:
top_triples = (
    triple_dist.groupby(["subject_label", "predicate", "object_label"], as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
    .head(25)
)
top_triples["triple"] = top_triples["subject_label"] + " → " + top_triples["predicate"] + " → " + top_triples["object_label"]
px.bar(top_triples, x="count", y="triple", orientation="h", title="Top relation triples").update_layout(yaxis={"categoryorder": "total ascending"}).show()

## 8. Validation Reports

These checks are the same idea as `make validate-all`: load the splits, deduplicate exact entity duplicates, and count offset validation failures.

In [ ]:
validation_reports = []
for quality in ALL_QUALITIES:
    report = build_validation_report(DATA_ROOT, quality)
    validation_reports.append({
        "split": quality,
        "articles": report["articles"],
        "entities": report["entities"],
        "duplicate_entities_removed": report["duplicate_entities_removed"],
        "relations": report["relations"],
        "offset_checks_passed": report["offset_checks_passed"],
        "offset_checks_failed": report["offset_checks_failed"],
        "missing_articles": report["missing_articles"],
    })
validation_df = pd.DataFrame(validation_reports)
validation_df

## 9. Dev Split Inspection

The dev split is the main local evaluation split because it includes gold entities and mention-level relations.

In [ ]:
dev = load_split(DATA_ROOT, "dev")
dev_articles = dev["articles"]
dev_entities = deduplicate_entities(dev["entities"])
dev_relations = dev["mention_relations"]

print("Dev articles:", len(dev_articles))
print("Dev entities:", len(dev_entities))
print("Dev mention-level relations:", len(dev_relations))

dev_articles.head()

In [ ]:
dev_entities.groupby("label").size().sort_values(ascending=False).rename("count").reset_index().head(20)

In [ ]:
dev_relations.groupby(["subject_label", "predicate", "object_label"]).size().sort_values(ascending=False).rename("count").reset_index().head(25)

## 10. Report-Ready Class Imbalance Summary

The generated Markdown summary is written to `outputs/reports/imbalance_summary.md` and can be reused directly in the course report.

In [ ]:
imbalance_path = REPORTS_DIR / "imbalance_summary.md"
imbalance_text = imbalance_path.read_text(encoding="utf-8")
display(Markdown(imbalance_text))

In [ ]:
summary_rows = []
for _, row in split_stats.iterrows():
    summary_rows.append(
        {
            "split": row["split"],
            "entity_majority": row["entity_majority_label"],
            "entity_majority_share": row["entity_majority_share"],
            "entity_imbalance_ratio": row["entity_imbalance_ratio"],
            "relation_majority": row["relation_majority_label"],
            "relation_majority_share": row["relation_majority_share"],
            "relation_imbalance_ratio": row["relation_imbalance_ratio"],
        }
    )
pd.DataFrame(summary_rows)

## 11. Takeaways

Use this section as a report checklist after running the notebook.

In [ ]:
gold_stats = split_stats[split_stats["split"] == "gold"].iloc[0]
dev_stats = split_stats[split_stats["split"] == "dev"].iloc[0]

print("Report-ready notes:")
print(f"- Gold train contains {int(gold_stats['documents'])} articles, {int(gold_stats['entities'])} entities, and {int(gold_stats['mention_level_relations'])} mention-level relations.")
print(f"- Dev contains {int(dev_stats['documents'])} articles, {int(dev_stats['entities'])} entities, and {int(dev_stats['mention_level_relations'])} mention-level relations.")
print(f"- Gold majority entity label is {gold_stats['entity_majority_label']} with share {gold_stats['entity_majority_share']:.3f}.")
print(f"- Gold majority relation predicate is {gold_stats['relation_majority_label']} with share {gold_stats['relation_majority_share']:.3f}.")
print("- Because label and predicate distributions are imbalanced, micro-F1 is the most stable headline metric, while macro-F1/per-label scores reveal minority-class failures.")

## 12. Generated Artifacts

The EDA stage produces these reusable report files:

```text
outputs/reports/data_stats_gold.csv
outputs/reports/data_stats_dev.csv
outputs/reports/data_stats_silver.csv
outputs/reports/data_stats_silver_2025.csv
outputs/reports/data_stats_bronze.csv
outputs/reports/entity_label_distribution.csv
outputs/reports/relation_label_distribution.csv
outputs/reports/relation_triple_distribution.csv
outputs/reports/entity_distribution.png
outputs/reports/relation_distribution.png
outputs/reports/imbalance_summary.md
```

<!-- notebook-output-images -->

### Saved model-comparison figures

These two figures summarize the final dev-set model comparison used in the project report.

![NER model F1 comparison](outputs/11_ner_model_f1_comparison.png)

![RE model F1 comparison](outputs/12_re_model_f1_comparison.png)
